In [1]:

import glob
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
from fugashi import Tagger


# ==========================================
# 1. 基本設定
# ==========================================

tagger = Tagger()

FUNCTION_POS = {"助詞", "助動詞", "接続詞", "連体詞"}
CONTENT_POS = {"名詞", "動詞", "形容詞", "形状詞", "副詞", "感動詞"}

base_path = r"C:\Users\Admin\2026-"

train_folder = os.path.join(base_path, "train_Human_text")
test_folder = os.path.join(base_path, "test_Human_text")


# ==========================================
# 2. Hapax率・Yule's K
# ==========================================

def hapax_yule(tokens, word_count):

    if word_count == 0:
        return 0, 0

    freq = Counter(tokens)

    # Hapax率
    hapax_count = sum(1 for v in freq.values() if v == 1)
    hapax_ratio = hapax_count / word_count

    # Yule's K
    freq_counter = Counter(freq.values())
    m1 = word_count
    m2 = sum((f ** 2) * c for f, c in freq_counter.items())

    yules_k = (
        10000 * ((m2 - m1) / (m1 * m1))
        if m1 > 0 else 0
    )

    return hapax_ratio, yules_k


# ==========================================
# 3. 文単位・累積特徴量抽出
# ==========================================

def analyze_text_files_per_sentence(folder_path):

    search_path = os.path.join(folder_path, "*.txt")

    filepaths = sorted(
        glob.glob(search_path),
        key=lambda x: (
            int(re.search(r"\d+", os.path.basename(x)).group())
            if re.search(r"\d+", os.path.basename(x))
            else 0
        )
    )

    feature_data = []

    for filepath in filepaths:

        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        # 文末記号を保持したまま文単位に分割
        sentences = re.findall(r"[^。！？]+[。！？]?", text)
        sentences = [s.strip() for s in sentences if s.strip()]

        basename = os.path.basename(filepath)

        # 1文目、1～2文目、1～3文目……と累積
        for s_idx in range(len(sentences)):

            sentence_text = "".join(sentences[:s_idx + 1])

            pos_counter = Counter()
            particle_counter = Counter()

            all_words = []
            content_words = []
            function_words = []

            word_count = 0
            content_count = 0
            function_count = 0

            # ----------------------------------
            # 文字・記号特徴量
            # ----------------------------------

            total_chars = len(sentence_text)

            if total_chars == 0:
                continue

            comma_count = sentence_text.count("、")
            period_count = sentence_text.count("。")
            exclamation_count = sentence_text.count("！")
            question_count = sentence_text.count("？")

            hiragana_count = len(re.findall(r"[ぁ-ん]", sentence_text))
            katakana_count = len(re.findall(r"[ァ-ン]", sentence_text))
            kanji_count = len(re.findall(r"[一-龥]", sentence_text))
            alnum_count = len(re.findall(r"[a-zA-Z0-9]", sentence_text))

            hiragana_ratio = hiragana_count / total_chars
            katakana_ratio = katakana_count / total_chars
            kanji_ratio = kanji_count / total_chars
            alnum_ratio = alnum_count / total_chars

            # ----------------------------------
            # 形態素解析
            # ----------------------------------

            for word in tagger(sentence_text):

                pos = word.feature.pos1
                surface = word.surface

                if pos == "補助記号":
                    continue

                if pos == "助詞":
                    particle_counter[surface] += 1

                all_words.append(surface)
                pos_counter[pos] += 1
                word_count += 1

                if pos in FUNCTION_POS:
                    function_words.append(surface)
                    function_count += 1

                elif pos in CONTENT_POS:
                    content_words.append(surface)
                    content_count += 1

            if word_count == 0:
                continue

            # ----------------------------------
            # 基本特徴量
            # ----------------------------------

            function_ratio = (function_count / word_count) * 100
            content_ratio = (content_count / word_count) * 100

            comma_ratio = comma_count / word_count
            period_ratio = period_count / word_count

            ttr = len(set(all_words)) / word_count

            # ----------------------------------
            # Hapax率・Yule's K
            # ----------------------------------

            hapax_all, yule_all = hapax_yule(
                all_words, word_count
            )

            hapax_func, yule_func = hapax_yule(
                function_words, function_count
            )

            hapax_cont, yule_cont = hapax_yule(
                content_words, content_count
            )

            # ----------------------------------
            # 助詞・品詞割合
            # ----------------------------------

            total_particle = sum(particle_counter.values())

            particle_ratio = {
                f"助詞_{k}": (
                    v / total_particle if total_particle > 0 else 0
                )
                for k, v in particle_counter.items()
            }

            pos_ratio = {
                pos: (count / word_count) * 100
                for pos, count in pos_counter.items()
            }

            # ----------------------------------
            # 特徴量を保存
            # ----------------------------------

            feature_data.append({

                "filename": f"{basename}_s{s_idx + 1}",

                "文長": total_chars,
                "総単語数": word_count,

                "機能語率": function_ratio,
                "内容語率": content_ratio,

                "異なり語数": len(set(all_words)),
                "TTR": ttr,

                "ひらがな率": hiragana_ratio,
                "カタカナ率": katakana_ratio,
                "漢字率": kanji_ratio,
                "英数字率": alnum_ratio,

                "Hapax率_all": hapax_all,
                "Yules_K_all": yule_all,

                "Hapax率_func": hapax_func,
                "Yules_K_func": yule_func,

                "Hapax率_cont": hapax_cont,
                "Yules_K_cont": yule_cont,

                "名詞割合": pos_ratio.get("名詞", 0),
                "動詞割合": pos_ratio.get("動詞", 0),
                "形容詞割合": pos_ratio.get("形容詞", 0),
                "形状詞割合": pos_ratio.get("形状詞", 0),
                "助詞割合": pos_ratio.get("助詞", 0),
                "助動詞割合": pos_ratio.get("助動詞", 0),
                "副詞割合": pos_ratio.get("副詞", 0),
                "連体詞割合": pos_ratio.get("連体詞", 0),
                "接続詞割合": pos_ratio.get("接続詞", 0),
                "感動詞割合": pos_ratio.get("感動詞", 0),

                **particle_ratio,

                "読点頻度": comma_ratio,
                "句点頻度": period_ratio,

                "感嘆符数": exclamation_count,
                "疑問符数": question_count,
            })

    return feature_data


# ==========================================
# 4. train / test の特徴量抽出
# ==========================================

train_data = analyze_text_files_per_sentence(train_folder)
test_data = analyze_text_files_per_sentence(test_folder)


# ==========================================
# 5. CSV出力
# ==========================================

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

train_df.to_csv(
    "train_Human_one_sentence_stylometry_features.csv",
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    "test_Human_one_sentence_stylometry_features.csv",
    index=False,
    encoding="utf-8-sig"
)


# ==========================================
# 6. 結果確認
# ==========================================

print("=== Train ===")
print(train_df.head())
print(f"総データ数：{len(train_df)}")

print("\n=== Test ===")
print(test_df.head())
print(f"総データ数：{len(test_df)}")


=== Train ===
               filename   文長  総単語数       機能語率       内容語率  異なり語数       TTR  \
0  Human_text_01.txt_s1   23    12  33.333333  66.666667     12  1.000000   
1  Human_text_01.txt_s2   73    36  41.666667  58.333333     25  0.694444   
2  Human_text_01.txt_s3  123    60  41.666667  58.333333     38  0.633333   
3  Human_text_01.txt_s4  197   101  45.544554  53.465347     54  0.534653   
4  Human_text_01.txt_s5  227   115  46.086957  53.043478     62  0.539130   

      ひらがな率     カタカナ率       漢字率  ...  助詞_へ  助詞_のみ  助詞_しも  助詞_しか  助詞_すら  助詞_より  \
0  0.347826  0.260870  0.304348  ...   NaN    NaN    NaN    NaN    NaN    NaN   
1  0.328767  0.342466  0.219178  ...   NaN    NaN    NaN    NaN    NaN    NaN   
2  0.325203  0.317073  0.211382  ...   NaN    NaN    NaN    NaN    NaN    NaN   
3  0.370558  0.299492  0.208122  ...   NaN    NaN    NaN    NaN    NaN    NaN   
4  0.356828  0.299559  0.229075  ...   NaN    NaN    NaN    NaN    NaN    NaN   

   助詞_こそ  助詞_つつ  助詞_くらい  助詞_かしら  
0 